# SSBG Raw Data File Exploration

In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/SSBG_dataset_2010-2022.csv')
# initial overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21240 entries, 0 to 21239
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   year                             21240 non-null  int64 
 1   state_name                       21240 non-null  object
 2   line_num                         21240 non-null  int64 
 3   service_category                 21240 non-null  object
 4   ssbg_expenditures                21240 non-null  object
 5   tanf_transfer_funds              21240 non-null  object
 6   total_ssbg_expenditures          21240 non-null  object
 7   other_fed_state_and_local_funds  21240 non-null  object
 8   total_expenditures               21240 non-null  object
 9   children                         21240 non-null  object
 10  adults_59_and_younger            21240 non-null  object
 11  adults_60_and_older              21240 non-null  object
 12  adults_unknown                  

Examine the df for cleaning and dtype optimization opportunities.

In [2]:
# Examine year column characteristics
print("Unique years:", sorted(df['year'].unique()))
print("Number of unique years:", df['year'].nunique())
print("Year value counts:")
print(df['year'].value_counts().sort_index())
print("\nCurrent memory usage of year column:", df['year'].memory_usage(deep=True), "bytes")

# Check if converting to category would save memory
print("Memory usage if converted to category:", df['year'].astype('category').memory_usage(deep=True), "bytes")

Unique years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Number of unique years: 13
Year value counts:
year
2010    1560
2011    1560
2012    1560
2013    1560
2014    1560
2015    1560
2016    1620
2017    1710
2018    1710
2019    1710
2020    1710
2021    1710
2022    1710
Name: count, dtype: int64

Current memory usage of year column: 170052 bytes
Memory usage if converted to category: 22032 bytes


## Category Columns

Note: Years 2010 to 2016 appear to have less rows than other years.
After converting, get a unique count for each category column to see how many unique values there are.

In [3]:
# convert ['year', 'state_name', 'line_num', 'service_category'] to category dtype
# print before and after memory savings for each column
columns_to_convert = ['year', 'state_name', 'line_num', 'service_category']
for col in columns_to_convert:
    before_mem = df[col].memory_usage(deep=True)
    df[col] = df[col].astype('category')
    after_mem = df[col].memory_usage(deep=True)
    print(f"'{col}': \nbefore conversion: {before_mem} bytes \nafter conversion: {after_mem} bytes")
    print(f"Categories for '{col}':", df[col].cat.categories.tolist(), end='\n\n')
    
           

'year': 
before conversion: 170052 bytes 
after conversion: 22032 bytes
Categories for 'year': [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

'state_name': 
before conversion: 1406592 bytes 
after conversion: 27277 bytes
Categories for 'state_name': ['Alabama', 'Alaska', 'American Samoa', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Guam', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Massachusetts Commission for the Blind', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Northern Mariana Islands', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Puerto Rico', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'U.S. Virgin Islands', 'Utah', 'Vermont', 'Vi

In [4]:
# examine the value counts for each category column after conversion and list any where the counts are not all the same
cols_not_uniform_value_counts = []
for col in columns_to_convert:
    value_counts = df[col].value_counts()
    if value_counts.nunique() > 1:
        cols_not_uniform_value_counts.append(col)

print("Columns with non-uniform value counts:", cols_not_uniform_value_counts)

#Show the value counts and highlight any values that are lower than the max
for col in cols_not_uniform_value_counts:
    print(f"Value counts for '{col}':")
    value_counts = df[col].value_counts().sort_index()
    max_count = value_counts.max()
    print(f"Max count for '{col}': {max_count}")
    for category, count in value_counts.items():
        if count < max_count: 
            print(f"  {category}: {count}")
    print()

Columns with non-uniform value counts: ['year', 'state_name']
Value counts for 'year':
Max count for 'year': 1710
  2010: 1560
  2011: 1560
  2012: 1560
  2013: 1560
  2014: 1560
  2015: 1560
  2016: 1620

Value counts for 'state_name':
Max count for 'state_name': 390
  American Samoa: 180
  Guam: 180
  Massachusetts Commission for the Blind: 180
  Northern Mariana Islands: 210
  U.S. Virgin Islands: 210



This seems clear that the US territories (e.g., American Samoa, Guam, Puerto Rico, and the USVI) and the Massachusetts Commission for the Blind have no data for some years. More investigation is needed to determine when data started being collected from these territories and the one oddball Org Mass-CFB. Do this in the clean data exploration notebook. 

## Int Columns

* These columns are dollar amounts. Some are $0 and some are "$1,345,654". These need to be cleaned and converted to int64 dtype.
1. 'ssbg_expenditures'
2. 'tanf_transfer_funds'
3. 'total_ssbg_expenditures', 
4. 'other_fed_state_and_local_funds', and 
5. 'total_expenditures'  

* These columns represent recipient counts. These need to be cleaned and converted to int64 dtype as well.
1. 'children'
2. 'adults_59_and_younger'
3. 'adults_60_and_older'
4. 'adults_unknown'
5. 'total_adults'
6. 'total_recipients'

In [5]:
# convert ['year', 'state_name', 'line_num', 'service_category'] to category dtype
# print before and after memory savings for each column
columns_to_convert = ['year', 'state_name', 'line_num', 'service_category']
for col in columns_to_convert:
    df[col] = df[col].astype('category')


# for cols ['ssbg_expenditures', 'tanf_transfer_funds', 'total_ssbg_expenditures', 'other_fed_state_and_local_funds', 'total_expenditures']
# remove the $ and , characters and convert to int dtype
dollar_columns = ['ssbg_expenditures', 'tanf_transfer_funds', 
                  'total_ssbg_expenditures', 
                  'other_fed_state_and_local_funds', 
                  'total_expenditures']
for col in dollar_columns:
    df[col] = df[col].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype('int64')

recipient_cols = ['children', 
                  'adults_59_and_younger', 'adults_60_and_older', 'adults_unknown',
                  'total_adults', 
                  'total_recipients']
for col in recipient_cols:
    df[col] = df[col].str.replace(',', '', regex=False).astype('int64')
    
df.info()
    

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21240 entries, 0 to 21239
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   year                             21240 non-null  category
 1   state_name                       21240 non-null  category
 2   line_num                         21240 non-null  category
 3   service_category                 21240 non-null  category
 4   ssbg_expenditures                21240 non-null  int64   
 5   tanf_transfer_funds              21240 non-null  int64   
 6   total_ssbg_expenditures          21240 non-null  int64   
 7   other_fed_state_and_local_funds  21240 non-null  int64   
 8   total_expenditures               21240 non-null  int64   
 9   children                         21240 non-null  int64   
 10  adults_59_and_younger            21240 non-null  int64   
 11  adults_60_and_older              21240 non-null  int64   
 12  adul

In [6]:
df.describe(include='all')

,year,state_name,line_num,service_category,ssbg_expenditures,tanf_transfer_funds,total_ssbg_expenditures,other_fed_state_and_local_funds,total_expenditures,children,adults_59_and_younger,adults_60_and_older,adults_unknown,total_adults,total_recipients
count,21240.0,21240,21240.0,21240,2.124000e+04,2.124000e+04,2.124000e+04,2.124000e+04,2.124000e+04,2.124000e+04,2.124000e+04,21240.000000,2.124000e+04,2.124000e+04,2.124000e+04
unique,13.0,57,30.0,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2017.0,Alabama,1.0,Administrative Costs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1710.0,390,708.0,708,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,9.866400e+05,7.002008e+05,1.686841e+06,1.791365e+07,1.960049e+07,7.094680e+03,2.990019e+03,1141.829896,4.296867e+03,8.428716e+03,1.552340e+04
std,NaN,NaN,NaN,NaN,4.993519e+06,6.938445e+06,9.920535e+06,1.862760e+08,1.922902e+08,7.553702e+04,5.387372e+04,14359.904396,1.065254e+05,1.258507e+05,1.739444e+05
min,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00
25%,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00
50%,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00
75%,NaN,NaN,NaN,NaN,1.599470e+05,0.000000e+00,2.487748e+05,5.564750e+04,9.507055e+05,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,6.500000e+02


This now appears to be cleaned and converted properly. We will export to a pickle.    

#TODO turn cleaning routine into a script.

In [ ]:
# df.to_pickle('../data/interim/ssbg_data_cleaned.pkl')